# 1. Free baselines with vLLM
Evaluates the base model on the full Spider dev set (1,034 questions) **without any API credit**: the model runs on the Colab GPU.

- Free T4: 1.5B or 3B.
- A100 (Colab Pro): up to 7B.

### Setup
Put `sql-agent-rl.zip` at the root of your Google Drive, or set `REPO_URL` to clone the GitHub repository instead. Models and runs are stored in Drive, so they survive the end of the Colab session.

In [ ]:
REPO_URL = ""  # empty: the code comes from MyDrive/sql-agent-rl.zip
from google.colab import drive
import os
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/sql-agent-rl"
if not os.path.exists("/content/sql-agent-rl"):
    if REPO_URL:
        !git clone -q $REPO_URL /content/sql-agent-rl
    else:
        !unzip -q /content/drive/MyDrive/sql-agent-rl.zip -d /content
%cd /content/sql-agent-rl
!pip install -q -e . -r train/requirements.txt
!python scripts/download_spider.py && python scripts/download_bird.py
!mkdir -p $DRIVE/outputs $DRIVE/runs && ln -sfn $DRIVE/outputs outputs && ln -sfn $DRIVE/runs runs

In [ ]:
import torch
print(torch.cuda.get_device_name(0))
DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "half"  # the free T4 has no bf16

In [ ]:
import subprocess, time, urllib.request

def serve(model, port=8000):
    """Start an OpenAI-compatible vLLM server and wait until it answers."""
    log = open("vllm.log", "w")
    proc = subprocess.Popen(["vllm", "serve", model, "--port", str(port), "--dtype", DTYPE,
                             "--max-model-len", "8192", "--gpu-memory-utilization", "0.9"], stdout=log, stderr=log)
    for _ in range(180):
        if proc.poll() is not None:  # crashed: no point waiting
            break
        try:
            urllib.request.urlopen(f"http://localhost:{port}/v1/models")
            print("vLLM ready:", model)
            return proc
        except Exception:
            time.sleep(5)
    proc.terminate()
    print("".join(open("vllm.log").readlines()[-40:]))
    raise RuntimeError("vLLM did not start: last lines of vllm.log above")

In [ ]:
MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
NAME = MODEL.replace("/", "_")
server = serve(MODEL)

### Without, then with the self-correction loop
`--max-turns 1`: a single reply. `--max-turns 4`: the model can explore, see its errors and fix them.

In [ ]:
!python -m sql_agent eval --base-url http://localhost:8000/v1 --split dev --workers 32 --model $MODEL --max-turns 1 --out runs/{NAME}_base_t1.jsonl
!python -m sql_agent eval --base-url http://localhost:8000/v1 --split dev --workers 32 --model $MODEL --max-turns 4 --out runs/{NAME}_base_t4.jsonl

In [ ]:
!python -m sql_agent compare runs/{NAME}_base_t1.jsonl runs/{NAME}_base_t4.jsonl

### Generalization: BIRD dev
Harder, larger databases, never seen in training. Each question comes with an evidence hint.

In [ ]:
!python -m sql_agent eval --base-url http://localhost:8000/v1 --split dev --workers 32 --dataset bird --model $MODEL --max-turns 4 --out runs/{NAME}_base_t4_bird.jsonl

### Optional: majority vote over 5 samples
Free here, but 5 times slower.

In [ ]:
!python -m sql_agent eval --base-url http://localhost:8000/v1 --split dev --workers 32 --model $MODEL --max-turns 4 --samples 5 --out runs/{NAME}_base_t4_k5.jsonl

In [ ]:
server.terminate()